# VWAP Statistical Model Research

## Phase 2 — Statistical Research Framework

### Research objectives

- Establish a reproducible historical research environment.
- Validate historical dataset quality and provenance.
- Reconstruct causal VWAP and probability-band observations.
- Reproduce the existing solid RMS and Gaussian ghost baselines.
- Evaluate statistical calibration and predictive performance.
- Establish chronological validation and out-of-sample protocols.

### Research principles

The existing engine is treated as a reference baseline rather
than an assumed optimal model.

Historical datasets must remain unchanged. Timestamp
conventions, overlapping observations, missing candles and
mixed-frequency data must be investigated before modelling.

Candidate distributions and trading rules must be evaluated
without look-ahead bias or retrospective parameter selection.

## 1. Research environment and dataset inventory

In [ ]:

from pathlib import Path
import sys

import pandas as pd
import numpy as np


# Locate the repository root whether the notebook starts
# from the project directory or the notebooks directory.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "research_config.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.research_config import RESEARCH_CONFIG


DATA_DIR = RESEARCH_CONFIG["data_dir"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Python interpreter: {sys.executable}")
print(f"Instrument: {RESEARCH_CONFIG['instrument']}")
print(f"Timeframe: {RESEARCH_CONFIG['timeframe']}")
print(f"Baseline: {RESEARCH_CONFIG['baseline_dataset']}")


In [ ]:

dataset_paths = sorted(DATA_DIR.glob("*.csv"))

assert dataset_paths, "No historical CSV datasets found."

dataset_inventory = pd.DataFrame(
    [
        {
            "filename": path.name,
            "size_mb": round(path.stat().st_size / 1024**2, 2),
            "columns": tuple(pd.read_csv(path, nrows=0).columns),
            "is_baseline": (
                path.name == RESEARCH_CONFIG["baseline_dataset"]
            ),
        }
        for path in dataset_paths
    ]
)

assert dataset_inventory["is_baseline"].sum() == 1, (
    "Expected exactly one baseline dataset."
)

print(f"Historical datasets: {len(dataset_inventory)}")
print(f"Total size: {dataset_inventory['size_mb'].sum():.2f} MB")

display(dataset_inventory)


## 2. Historical Dataset Registry and Provenance

Historical observations are separated into three categories:

- Legacy historical datasets retained for baseline reproduction
  and comparison.
- Immutable raw MT5 acquisitions with recorded source metadata
  and content-derived dataset identities.
- Validated research datasets produced only after independent
  timestamp and data-quality assessment.

External storage allows datasets to be shared across research
branches without duplication or inclusion in Git history.

Successful file-integrity verification does not establish
historical timezone accuracy, trading-session completeness
or eligibility for statistical modelling.

In [ ]:
from src.research_data_diagnostics import sha256_file

legacy_dataset_registry = pd.DataFrame(
    [
        {
            "filename": path.name,
            "dataset_type": "legacy_historical",
            "dataset_id": f"sha256:{sha256_file(path)}",
            "file_size_bytes": path.stat().st_size,
            "is_baseline": (
                path.name == RESEARCH_CONFIG["baseline_dataset"]
            ),
            "provenance_status": "not_independently_verified",
            "research_eligibility": "pending_validation",
        }
        for path in dataset_paths
    ]
)

display(legacy_dataset_registry)

In [ ]:
from src.research_dataset_registry import (
    discover_raw_datasets,
)

RAW_DATA_DIR = RESEARCH_CONFIG["raw_data_dir"]
VALIDATED_DATA_DIR = RESEARCH_CONFIG["validated_data_dir"]

print(f"External raw data: {RAW_DATA_DIR}")
print(f"Validated data: {VALIDATED_DATA_DIR}")

raw_dataset_records = discover_raw_datasets(
    RAW_DATA_DIR
)

raw_dataset_registry = pd.DataFrame(
    raw_dataset_records
)

print(
    f"Registered raw acquisitions: "
    f"{len(raw_dataset_registry)}"
)

if not raw_dataset_registry.empty:
    display(
        raw_dataset_registry[
            [
                "dataset_label",
                "symbol",
                "timeframe",
                "rows",
                "broker_server",
                "integrity_verified",
                "timestamp_alignment",
                "research_eligibility",
            ]
        ]
    )
else:
    print("No raw acquisitions registered.")

In [ ]:
if not raw_dataset_registry.empty:
    display(
        raw_dataset_registry[
            [
                "dataset_label",
                "dataset_id",
                "requested_start",
                "requested_end_exclusive",
                "observed_first_timestamp",
                "observed_last_timestamp",
                "acquisition_started_utc",
                "code_version",
                "working_tree_dirty",
            ]
        ]
    )